# Solo-buildable iOS idea scoring audit

Recomputes v4.0 archetype-weighted low/base/high scores and desk decisions for the three shortlisted ideas. Inputs are evidence-bounded judgments, not forecasts.

In [1]:
from decimal import Decimal, ROUND_HALF_UP

WEIGHTS = {
    'consumer_utility': {
        'problem_demand': 25, 'monetization_economics': 15,
        'competitive_wedge': 20, 'distribution_access': 25,
        'repeat_value_retention': 5, 'timing': 10,
    },
    'b2b_prosumer': {
        'problem_demand': 25, 'monetization_economics': 25,
        'competitive_wedge': 15, 'distribution_access': 15,
        'repeat_value_retention': 15, 'timing': 5,
    },
}

IDEAS = {
    'Quilt Layout Lock': {
        'archetype': 'consumer_utility',
        'scores': {
            'problem_demand': (3, 4, 4), 'monetization_economics': (2, 3, 3),
            'competitive_wedge': (3, 3, 3), 'distribution_access': (3, 3, 3),
            'repeat_value_retention': (2, 3, 3), 'timing': (2, 3, 3),
        },
        'evidence': 'C', 'gates': ['clear'] * 4,
    },
    'Resin Calibration Ladder': {
        'archetype': 'consumer_utility',
        'scores': {
            'problem_demand': (3, 4, 4), 'monetization_economics': (2, 3, 3),
            'competitive_wedge': (3, 3, 3), 'distribution_access': (3, 3, 3),
            'repeat_value_retention': (2, 3, 3), 'timing': (3, 3, 3),
        },
        'evidence': 'C', 'gates': ['clear'] * 4,
    },
    'Laser Scrap Passport': {
        'archetype': 'b2b_prosumer',
        'scores': {
            'problem_demand': (3, 3, 4), 'monetization_economics': (2, 3, 3),
            'competitive_wedge': (2, 3, 3), 'distribution_access': (3, 3, 3),
            'repeat_value_retention': (3, 3, 3), 'timing': (3, 3, 3),
        },
        'evidence': 'C', 'gates': ['clear'] * 4,
    },
}


In [2]:
SCENARIOS = ('low', 'base', 'high')

def round_half_up(value):
    return int(Decimal(str(value)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def opportunity_interval(spec):
    weights = WEIGHTS[spec['archetype']]
    totals = []
    for scenario_index in range(3):
        raw = sum(weights[d] * spec['scores'][d][scenario_index] / 5 for d in weights)
        totals.append(round_half_up(raw))
    return tuple(totals)

def classification(interval):
    low, _, high = interval
    if high < 40:
        return 'weak'
    if low >= 60:
        return 'attractive'
    return 'borderline'

def desk_decision(interval, evidence, gates):
    low, _, high = interval
    if 'blocked' in gates:
        return 'ELENDIR'
    if 'unknown' in gates or evidence == 'D':
        return 'BELIRSIZ'
    if high < 40:
        return 'ELENDIR'
    if low >= 60:
        return 'TEST_ONCELIKLI'
    return 'TESTE_DEGER'


In [3]:
expected = {
    'Quilt Layout Lock': (54, 65, 65),
    'Resin Calibration Ladder': (56, 65, 65),
    'Laser Scrap Passport': (52, 60, 65),
}

rows = []
for name, spec in IDEAS.items():
    interval = opportunity_interval(spec)
    assert interval == expected[name], (name, interval, expected[name])
    cls = classification(interval)
    decision = desk_decision(interval, spec['evidence'], spec['gates'])
    assert cls == 'borderline'
    assert decision == 'TESTE_DEGER'
    rows.append((name, spec['archetype'], *interval, cls, spec['evidence'], decision))

headers = ('idea', 'archetype', 'low', 'base', 'high', 'class', 'evidence', 'decision')
print(' | '.join(headers))
print(' | '.join(['---'] * len(headers)))
for row in rows:
    print(' | '.join(map(str, row)))
print('\nPASS: arithmetic, classification, and desk-decision assertions')


idea | archetype | low | base | high | class | evidence | decision
--- | --- | --- | --- | --- | --- | --- | ---
Quilt Layout Lock | consumer_utility | 54 | 65 | 65 | borderline | C | TESTE_DEGER
Resin Calibration Ladder | consumer_utility | 56 | 65 | 65 | borderline | C | TESTE_DEGER
Laser Scrap Passport | b2b_prosumer | 52 | 60 | 65 | borderline | C | TESTE_DEGER

PASS: arithmetic, classification, and desk-decision assertions
